# Ensemble learners and Feature Selection 


In this exercise we try to predict water consumption in North America. 
For this we use a subset of a larger dataset consisting of 800.000 labeled end-use events from 762 homes across the USA and Canada. For licensing and privacy reasons, we have modified the subset in certain aspects so that it does not accurately reflect the water consumption of the surveyed households.\ 
Our aim is to predict the different types of water use in private households. So, we assume that various features, such as the time of the day, the volume and duration of water use, provide enough information to estimate in which way the water is used (e.g. for showers, dishwasher, pool or irrigation).


Besides testing the performance of different ensemble learners, we will also have a look into filtering, feature engineering, and of course some common importance measures. 
For the software development-part in this exercise, we will use a simple Pipeline from `sklearn` and will later conduct some experimental testing in [Weights and Biases](https://wandb.ai/site/).



The exercise is based on a recent study from Gross et al., 2025: *A Machine Learning-based framework and open-source software for Non Intrusive Water Monitoring*  \
----> [Here it goes to the study](https://doi.org/10.1016/j.envsoft.2024.106247)


**Have a happy coding time** :)

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
import wandb

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.inspection import permutation_importance   



## Weights & Biases (W&B)
In this notebook the models are tracked in their performances by using the toolkit from Weights & Biases. For further information how to use W&B for experimental testing, follow [this link]( https://docs.wandb.ai/models/integrations/scikit#python-notebook)

Uncomment the code in the code cell below, to login W&B and to initialize a new run

In [ ]:
# wandb.login()

# wb_run = wandb.init(project="ai4hws", name="water-use-clf")

## Dataset

Load in both datasets and conduct a short exploratory analysis. \
Check for the number of features. Are their scales nominal, ordinal or intervals? 
Also, check for potential missing values in the feature space.\
Plot the distribution of the target variable, we want to predict (`SumAs`). What do you see in the histogram? Could the distribution be an issue for the models?




In [ ]:
session_name = "05_session"

root = Path(os.getcwd()).parent
data_dir = Path(f"{root}/{session_name}/data")


In [ ]:
df_test_org = pd.read_csv(data_dir / 'testData_modified.csv')
df_train_org = pd.read_csv(data_dir / 'trainData_modified.csv')

In [ ]:
print(df_test_org.shape)
print(df_train_org.shape)

In [ ]:
df_train_org.tail(10)

In [ ]:
df_train_org.SumAs.hist(
    bins=len(df_train_org.SumAs.unique())-1,
    xrot=90
)

From the distribution of the target variable, we can see tha 4 categorise are quite common in the target variable.
In regard to their occurrence, the 4 categorise reflect the most common types of water consumption in households and each time when water runs through a pipe, a small share of it will lea out.\
For the classification models, it means that they will be trained on those 4 common categorise much more than on more seldom categorise, such as cases of water usage for irregation.\
What we see are imbalanced classes in the target variable, which is a quite common case in ML. Usually, we would conduct one of the resampling techniques, such as oversampling or undersampling. In addition, it could be useful to adjust the class weights. When the dataset is very small and the imbalance strong, it can be useful to only adapt the class weights (without resampling) in the way that during model training the algorithm is penalized more on the underrepresented class (minority) than on the overrepresented class (majority) leading to higher weights (i.e. model parameters) o the underrepresented class. For instance, this was done to predict the probability of having economic losses due to a flood with a logistic regression (https://nhess.copernicus.org/articles/25/2437/2025/)

### Task: 

1. For having a better visualization of the relationships between features and target variable, we just pick a random subset of 20 % form our larger subset.
2. Then, we want to visualize how long and much water is used by the different usage types (e.g. shower, leakage, irrigation, etc.). For this subtask, group the records based on the column `SumAs` which refers to the usage type of the water. After grouping, plot the feature for the time duration of the water usage (`Duration`) against the used water volume (`Volume`).  \
**Hint:** You can do this quite straight forward by creating a group of plots - called facets - using the seaborn functions `FacetGrid()` and `map_dataframe()`.\
3. Have a look at the scatterplots - which interesting aspects or patterns do you see? 


In [ ]:
# 1.
df_test = df_test_org.sample(frac=0.2, replace=True, random_state=1).reset_index(drop=True)
df_train = df_train_org.sample(frac=0.2, replace=True, random_state=1).reset_index(drop=True)
df_train.tail()

In [ ]:
# 2.
g = sns.FacetGrid(df_test, col="SumAs", sharex = False, sharey = False, col_wrap = 4)

g.map_dataframe(sns.scatterplot, x="Duration", y="Volume")
g.set_axis_labels("Duration", "Volume")
g.add_legend()

3. 
We see many things, for example, ..
- The most of the time and volume water is used for bathroom and kitchen
- The water usage of some devices (e.g. dishwasher vs clotheswasher) differs remarkably
- Some of the devices or facilities are not very water-saving. Probably they are quite old
- The leakage is quite constant over time
- We have for most distributions quite some outliers, which does not necessarily mean that these datapoints are erroneous values

# Feature Engineering

## Label Encoding
We want to see how much impact the information about the weekday has in our ML models. However, at the moment the information about the weekdays is spread across multiple columns. Thus, we need to combine the binary variables for the 7 weekdays into one "ordinal" scaled feature. Dont forget to drop the old columns for the single weekdays in the new dataframe.

Then, convert the categorize ("weekday_Friday", "Weekday_Monday"..) into ordinal scale by ranking them based on numbers.


In [ ]:
# 1.
all_weekdays = ["Weekday_Friday", "Weekday_Monday" ,"Weekday_Saturday","Weekday_Sunday","Weekday_Thursday","Weekday_Tuesday","Weekday_Wednesday"]
df_train["Weekday"] = df_train[all_weekdays].idxmax(axis=1)
df_test["Weekday"] = df_test[all_weekdays].idxmax(axis=1)

df_train.drop(columns=all_weekdays, inplace=True)
df_test.drop(columns=all_weekdays, inplace=True)
df_train.tail(3)


In [ ]:
# 2. 
df_train = df_train.replace(all_weekdays,[5,1,6,7,4,2,3])
df_test = df_test.replace(all_weekdays,[5,1,6,7,4,2,3])
df_train["Weekday"] = df_train["Weekday"].astype(int)
df_test["Weekday"] = df_test["Weekday"].astype(int) 
df_train.head()

In [ ]:
print(df_test.shape)
print(df_train.shape)

### Trainings set and test sets  

Our target variable is the type of water use (`SumAs`), which means we are dealing with a classification task. We separate the target variable from the predictor candidates (i.e. independent variables), so that we get four dataframes: `X_train, y_train, X_test, y_test`  

In [ ]:

# split into input (X) and output (y) variables
X_train = df_train.drop(["SumAs"], axis = 1)
X_test = df_test.drop(["SumAs"], axis = 1)

y_train = df_train["SumAs"]
y_test = df_test["SumAs"]


In [ ]:
print('Train', X_train.shape, y_train.shape)
print('Test', X_test.shape, y_test.shape)

### Some more Label Encoding

As a next step, we recode the names of our water usage categories (i.e., target variable). For this, we use the `LabelEncoder()` function from `sklearn`. Write a simple function for this task, which calls the encoder, fits it based on the values of the target variable in the training set, and use it to rescale them as well as the target variable in the test set. 
Think about (or do some small internet search) why the Encoder should be only fitted on the training set and not on the test set?


In [ ]:
# prepare target and return encoded training and testing data
def prepare_targets(y_train, y_test) -> tuple[np.ndarray, np.ndarray]:
    
    le = LabelEncoder()
    
    # 1. Approach
    # le.fit(y_train) 
    # y_train_enc = le.transform(y_train)
    # y_test_enc = le.transform(y_test)
    
    # 2. Approach - more safer (in my opinion)
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)
    
    return y_train_enc, y_test_enc

In [ ]:
y_train_enc, y_test_enc = prepare_targets(y_train, y_test)
y_train_enc

2. Why we should fit only one encoder and not two encoders - one for the test set and one on the training set? \
Note: this applies also for encoding features, or standarization (e.g. with MinMaxScaler)

* In short, it could be that the test set contains other values than the training set, e.g. rarer values which not occur in the training set, but only in the test set. Fitting two encoders and transforming the sets separately would create new relationships between the target variable and its features in each set. Applying a model, trained only on the relationships in the training set, then on the test set could drop its performance as it has not seen the new relationships between target and features before.

* Also applying an encoder or scaler on all records of a dataset is not a good practice, because we should ensure in anyway that no information about the test data leaks into the training set. Fitting an encoder or scaler on all data and then transforming it, would cause that the training set would be transformed based also on the information of the test samples and not only the training samples.


### Further reading: 
Check the coursebook for more information when categorical variables should be encoded (Coursebook page: "Encoders"). 

# Analysing correlations in the dataset

<!-- ### Variance Threshold


In this step, we want to analyse which of the temporal features in our dataset contain potentially unimportant information. Usually the more variance a feature contains, the more it potentially can contribute to the model training (e.g. when the cost function is minimized by adjusting the weights, also called model parameters). \
This is a quite common case, as an AI user we can only assume in the beginning of a project that certain features would contain important information and, thus, are useful for our modelling task. However, when a feature as variance close to 0 then it probably is not useful for improving our model. -->

<!-- The Variance describes how far the values spread around the mean value.
$$Var(x)^2 =  \displaystyle \frac {\sum (x_{i} - \mu)^2 }{n}$$

It can also be described as the squared standard deviation.
$$ Var(x) = \sigma ^2 $$

|   |   |   
|---|---|
| $\mu $  | mean value of the sample|  
| $x_i$  | observation  |  
| x  | sample distribution  | 
| n   | number of observations  |   
| $\sigma$   | standard deviation  |   -->


In [ ]:
# a = np.random.normal(2, .3, 1000)
# b = np.random.normal(2, 1, 1000)

# concatenated = np.core.records.fromarrays([a,b],names='high variance,low variance')

# sns.displot(concatenated)

<!-- If the Variance of a Feature is close to 0 it must not be used in modelling later on and can be excluded immediately. -->

In [ ]:
# selector = VarianceThreshold(threshold=0.01) # Variance threshold 
# sel = selector.fit(X_train)
# sel_index = sel.get_support()
# X_train_vt = X_train.iloc[:, sel_index]
# print(X_train_vt.columns)

# print(len(X_train_vt.columns) == len(X_train.columns))

### Find out which features could be potentially important for predicting the water usage type
We do this by analyzing the correlations between features and between features and the target.
For measuring the correlation strength, we use the Spearman rank correlation coefficient (\(\rho \) or \(r_{s}\)) which measures the strength and direction of a monotonic relationship between two ranked or ordinal variables. It is more suitable for our dataset than the Pearson Correlation, which measures only linear relationships.

Simply uncomment the plot function below and apply it to analyse the Spearman rank correlation.

In [ ]:
import pandas as pd
from scipy.stats import spearmanr, skew, norm


def plot_single_corr_matrix(df, ignore=None, alpha_scatter=0.5, cmap='coolwarm', significance_levels=None):

    # Pre-filter numerical columns
    df_numeric = df.select_dtypes(include='number').copy()
    if ignore:
        df_numeric.drop(columns=ignore, inplace=True, errors='ignore')
    df_numeric = df_numeric.dropna()  # Drop rows with any NaN once

    features = df_numeric.columns.tolist()
    n = len(features)

    # Prepare significance levels
    sig_items = sorted(significance_levels.items(), key=lambda x: x[0]) if significance_levels else []

    # Precompute correlation and p-value matrices
    corr_matrix = np.zeros((n, n))
    pval_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            corr, pval = spearmanr(df_numeric.iloc[:, i], df_numeric.iloc[:, j])
            corr_matrix[i, j] = corr
            pval_matrix[i, j] = pval
            corr_matrix[j, i] = corr  # symmetric
            pval_matrix[j, i] = pval

    _, axes = plt.subplots(n, n, figsize=(3 * n, 3 * n))  # _  means we dont need this variables, thus it got no name
    plt.subplots_adjust(wspace=0.3, hspace=0.3)

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            feat_i = features[i]
            feat_j = features[j]
            corr = corr_matrix[i, j]
            p_val = pval_matrix[i, j]
            color = plt.cm.get_cmap(cmap)((corr + 1) / 2)

            if i == j:
                data = df_numeric[feat_i]
                ax.hist(data, bins=30, density=True, edgecolor='black', color="grey")
                mean_val, std_val = np.mean(data), np.std(data)
                x_vals = np.linspace(data.min(), data.max(), 100)
                ax.plot(x_vals, norm.pdf(x_vals, mean_val, std_val), color='red', linewidth=2)
                skewness_val = skew(data)
                ax.text(0.95, 0.95, f"skew={skewness_val:.2f}", transform=ax.transAxes,
                        ha='right', va='top', fontsize=10, color='blue')
                ax.set_xlabel(feat_i) if i == n - 1 else ax.set_xticks([])
                ax.set_ylabel('Density') if j == 0 else ax.set_yticks([])
            elif i > j:
                ax.set_facecolor(color)
                x = df_numeric[feat_j].values
                y = df_numeric[feat_i].values
                ax.scatter(x, y, s=10, alpha=alpha_scatter, color='black')
                if len(x) > 1:
                    m, b = np.polyfit(x, y, 1)
                    ax.plot([x.min(), x.max()], [m * x.min() + b, m * x.max() + b], color='red', linewidth=1)
                ax.set_xlabel(feat_j) if i == n - 1 else ax.set_xticks([])
                ax.set_ylabel(feat_i) if j == 0 else ax.set_yticks([])
            else:
                ax.set_facecolor(color)
                sig_marker = ''
                for thr, marker in sig_items:
                    if p_val < thr:
                        sig_marker = marker
                        break
                ax.text(0.5, 0.5, f"r={corr:.2f}\np={p_val:.3f}{sig_marker}", va='center', ha='center', fontsize=12)
                ax.set_xticks([])
                ax.set_yticks([])

            ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
significance_levels = {0.001: '***', 0.01: '**', 0.05: '*'}

# small fix onthe fly. make the "weekdays" as integers
df_train["Weekday"] = df_train["Weekday"].astype(int)
df_test["Weekday"] = df_test["Weekday"].astype(int)

plot_single_corr_matrix(df_train, alpha_scatter=0.3, significance_levels=significance_levels)

# Train multiple ensemble learners 

Your task is it to train at least two common ensemble models and to compare their performances 

* Pick, for instance, at least two ensemble learners from [sklearn](https://scikit-learn.org/stable/api/sklearn.ensemble.html), a tree-based [LightGBM](https://lightgbm.readthedocs.io/en/stable/) or [XGBoost](https://xgboost.readthedocs.io/en/stable/) model.
* Add the most important hyperparameters and value ranges you want to test to `hyperparameter_dict`
* Is any of the models suitable to handle missing values in the training data? 

Uncomment the code cell below and add your selected models, hyperparameters and hyperparameter values

In [ ]:
# classifiers = [
#     Classifier_1(),
#     Classifier_2(),
#     ...
# ]


# hyperparameter_dict = {

#     "XGBClassifier_hyperparameters":{
#         "model__hyperparameter_1": ,
#         "model__hyperparameter_2": 
#         "...": 
#     },
#     "AdaBoostClassifier_hyperparameters":{
#     "model__hyperparameter_1": ,
#     "model__hyperparameter_2": 
#     "...": 
#     },
#     ....
# }

In [ ]:
## We test following models to predict the type of water usage:

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier


classifiers = [
    RandomForestClassifier(),
    XGBClassifier(),   # Anna's favorite
    AdaBoostClassifier(),
    LGBMClassifier(),
]

# and the models hyperparameters to test
hyperparameter_dict = {

    "XGBClassifier_hyperparameters":{
        "model__n_estimators": [ 3, 5, 10, 15, 30, 50, 100, 200 ],
        "model__max_depth": [ 3, 5, 7, 10, 15],
        "model__colsample_bytree": [0.33, 0.66, 1.0],
        "model__eta": [0.001, 0.01, 0.1, 0.2, 0.5],
    },
    "LGBMClassifier_hyperparameters":{
        "model__n_estimators": [3, 5, 10, 15, 30, 50, 100, 200],
        "model__max_depth": [ 3, 5, 7, 10, 15],
        "model__num_leaves": [3, 5, 7, 10, 15]
    },
    "RandomForestClassifier_hyperparameters":{
        "model__n_estimators": [3, 5, 10, 15, 30, 50, 100, 200],
        "model__max_depth": [ 3, 5, 7, 10, 15],
        "model__max_features": [0.33, 0.66, 1.0],    
    },
    "AdaBoostClassifier_hyperparameters":{
        "model__n_estimators": [3, 5, 10, 15, 30, 50, 100, 200],
        "model__learning_rate": [0.001, 0.01, 0.1, 0.5, 1.0, 2.0],  
    },
}

**Create a train-test-evaluate function:**
*  We will have to carry out the steps of training, predicting, and evaluating an estimator multiple times, so let's just create a function for this. It should take an estimator instance, the complete dataset as input, and should return four different performance measures - choose a suitable averaging method for certain performance measures to account for the class-imbalance in the target variable (**Hint**: Use the "?" to read about the parameters needed for a (skearn) function, e.g. `precision_score?`). Return also the predicted and observed type of water use for the test set (i.e. y_pred_enc, y_test_enc). Use the encoded target variables. In addition, store the final models in a list, we need them later for the feature importance measures
* In addition, this function should remove samples only for models which can not handle missing values. Cross-validate your models and use if possible a `Pipeline` from `sklearn.pipeline` in the function. As our dataset does not contain any missing samples we can exceptionally skip this step when writing the function.

**Hint**: You can write your own function, or uncomment the code in the second cell below for some guidance.

**Note**
- Make sure to use only features which make sense, i.e. which might contribute to the model learning
- It is sometimes needed to scale your X_train and X_test data e.g. via a `MinMaxScaler()`. THe purpose is that all features are getting scaled between 0 and 1 which usually is beneficial for model learning. However,for tree-based models a scaling as preprocessing step is not mandatory

In [ ]:
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# def tpe(clf:str, X_train:pd.DataFrame, y_train:np.ndarray, X_test:pd.DataFrame, y_test:np.ndarray, hyperparameter_dict:dict):


    # get the name of the tested model, use type(classifier_name).__name__


    # set up pipeline with sklearn.pipeline
    # pipe = Pipeline(steps=[("model", clf),])


    # print(f"Use {X_train.shape[0]} samples for training, and {X_test.shape[0]} samples for evaluation of {model_name}.")

    # setup the randomized search cross-validation
    # clf_cv = RandomizedSearchCV(
    #     pipe, 
    #     hyperparameter_dict[f"{model_name}_hyperparameters"], 
    #     scoring="<place a scoring metric here>",
    #     n_iter="<decide for a suitable number of iterations, ie. number of tried parameter settings >", 
    #     cv="<decide for a suitable number of folds>",
    #     random_state=42,  
    # )

    # conduct the model training with hyperparameter tuning and save the best estimated resulted from CV

    # predict with the best estimator on hold-out test set

    # calculate performance metrics (account for the class imbalance by taking a suitable average method), 

    # return the four performance measures, the predicted classes and the best estimator


In [ ]:

# use a decorator to define how the output of tpe() should look
@dataclass
class performance_result:
    acc: float
    precision: float
    recall: float
    f1_micro: float
    y_pred: np.ndarray
    best_estimator: RandomizedSearchCV


def tpe(
        clf:str, 
        X_train:pd.DataFrame, y_train:np.ndarray, 
        X_test:pd.DataFrame, y_test:np.ndarray, 
        hyperparameter_dict:dict
    ) -> performance_result:

    ## NOTE: see info about function with ?tpe

    # get the name of the tested model
    model_name = type(clf).__name__

    # set up pipeline with skleanr.pipeline
    pipe = Pipeline(steps=[("model", clf),])


    print(f"Use {X_train.shape[0]} samples for training, and {X_test.shape[0]} samples for evaluation of {model_name}.")


    clf_cv = RandomizedSearchCV(
        estimator=pipe, 
        param_distributions=hyperparameter_dict[f"{model_name}_hyperparameters"], 
        scoring="f1_micro", 
        n_iter=3, 
        cv=5,
        random_state=42,  
        verbose=1
    )

    best_estimator = clf_cv.fit(X_train, y_train) 
    y_pred = best_estimator.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="micro")
    recall = recall_score(y_test, y_pred, average="micro")
    f1_micro = f1_score(y_test, y_pred, average="micro")


    return performance_result(acc=acc, precision=precision, recall=recall, f1_micro=f1_micro, y_pred=y_pred, best_estimator=best_estimator)


### Model comparison

Iterate over the `classifiers` and find out which one performed best in predicting the type of water use.

Tasks:
- Create a loop which iterates over the classifiers and apply your train-test-evaluate function (optional: write out how long the training and evaluation takes for each model. This is just for practice).
In a larger project we would now write out all tested models (e.g as pickle files) and store their predictions, but for now we can also go the lazy way and only focus on the best performed model 
- You can identify your best model simply by looking at the performance measure which was used as scoring method during Cross-validation (in case CV was used)
- Which model performed well and for which classes?

In [ ]:

y_pred_list = []
final_models_list = []
df_final_performances = pd.DataFrame()

## apply train-predict-evaluate function
for clf in classifiers:

    model_name = type(clf).__name__
    print(f"\n--------- {model_name} -----------")
    TIME0 = datetime.now()

    # train, predict and evaluate each model
    performance = tpe(
    ## acc, precision, recall, f1_micro, y_pred, best_estimator = tpe(
        clf, 
        X_train, y_train_enc, 
        X_test, y_test_enc, 
        hyperparameter_dict,
    )
    print(f"Training and evaluation of {model_name} took {np.round((datetime.now()-TIME0).total_seconds(), 1)} seconds \n")

    # save the predictions and the test set, as well as the best model
    df_performance_scores = pd.DataFrame(
        {"model": model_name,
         "acc": performance.acc,
         "precision": performance.precision,
         "recall": performance.recall,
         "f1_micro": performance.f1_micro
        }, index=[0]
    )
    df_final_performances = pd.concat([df_final_performances, df_performance_scores], ignore_index=True)

    y_pred_list.append(performance.y_pred)
    final_models_list.append(performance.best_estimator)
    


In [ ]:
# 3. best performed model
df_final_performances


3. 
The models performed all very similar. We could try to improve the hyperparameters, but probably we would just gain a few more points in the performances - no remarkable improvement. 
Based on the very similar predictive performances, we can take any model we want for inference, e.g. to regional transfer it to another study side.
In the steps below the best estimator from the XGBoost model is used (but any model would be equally suitable). The aim is to see which features are important for predicting on unseen samples (i.e., the test set)

# W&B: Analyse prediction results 

### Task:
Fo an easier start you can simply pick any model from the list of trained models. In the example code cell below, the XGB model was selected. For the ROC curve the probabilistic estimates are needed. You can simply can generate them by using `predict_proba()` as shown in the code cell below. They indicate the likelihood to which class a specific datapoint (i.e. sample) belongs.
* Try out W&B for experimental testing and also create plots with W&B showing a confusion matrix and the class proportions of your target variable in the training and testing set
* Plot also the ROC and learning curves with W&B


NOTE: the plots are shown on your dashboard in W&B\
Further examples: https://colab.research.google.com/github/wandb/examples/blob/master/colabs/scikit/Simple_Scikit_Integration.ipynb#scrollTo=Nlz9Wgu-Twoq



In [ ]:
# pick any model (here XGB)
best_estimator_xgb = final_models_list[0]
labels = y_test.values


# get predictions and their probabilities (i.e. how sure the model is in its prediction)
# needed for ROC
y_pred = best_estimator_xgb.predict(X_test)
y_proba = best_estimator_xgb.predict_proba(X_test)
y_proba



In [ ]:
## Plot Confusion matrix with WB



In [ ]:
# Class proportions


In [ ]:
## ROC


In [ ]:
## Learning curve


### Task
* W&B has many more functionalities than the ones we tried out before. Just spend some more time and get more familiar with some of those other functionalities.Share your experiences or results with the rest of the course

# Calculate Feature Importance

Several types of feature importance measure exist, e.g. impurity-based feature importance (call by `final_model.feature_importances_`) or the permutation-based feature importance.
For now, we go with the permutation-based measure..


We are interested in the feature importances for the model predictions, thus we apply the measure on our test set. The importance scores represent then the increase in the model error when doing predictions.


## *Permutation based feature importance*

Permutation feature importance overcomes limitations of the impurity-based feature importance: they do not have a bias toward high-cardinality features and can be computed on a left-out test set.

Recommended further reading: [scikit-documentation](https://scikit-learn.org/stable/modules/permutation_importance.html#permutation-importance)

**Important notice:** When two features are correlated and one of the features is permuted, the model will still have access to the feature through its correlated feature. This will result in a lower importance value for both features, where they might actually be important.

1.&emsp;fitted predictive model $m$, tabular dataset (training or validation) $D$

2.&emsp;Compute the reference score $s$ of the model $m$ on data $D$ (for instance the accuracy for a classifier or the $R^2$ for a regressor)

3.&emsp;For each Feature j (column of $D$) <br>
&emsp;&emsp;(a)&emsp;for each repetition $ k $ in $ 1,..., K$ <br>

&emsp;&emsp;&emsp;$\cdot$&emsp;Randomly shuffle column $j$ of dataset $D$ to generate a corrupted version of the data $D_{k,j}$<br>

&emsp;&emsp;&emsp;$\cdot$&emsp; compute the score $s_{k,j}$ of model m on corrupted data $D_{k,j}$<br><br>

&emsp;&emsp;(b)&emsp;Compute importance $i_j$ for feature $f_j$ defined as:<br><br>
    $$ i_j = s- \frac{1}{K} \sum s_{k,j}$$


### Task: 
- Apply a permutation-based feature importance on the best estimator on one of the models evaluated before. The aim is to determine the mean decrease of a prediction result when the links between a certain feature and the target are remove (i.e. the feature is left out). For now we want to return only the averaged importance score per feature. Decide which `scoring` mechanism would suit best to the data, consider to use a `seed` in order to obtain always the same values in the importance scores. **Hint:** The accuracy as scoring method is maybe not the best choice ;)
- Plot the importance scores


In [ ]:
# NOTE y_pred_list[0] contains the predicted samples from XGBoost (the best performed model), the same applies to final_models_list[0]

best_estimator_xgb = final_models_list[0]
y_pred_xgb =  y_pred_list[0]

In [ ]:

fi_mean = permutation_importance(
        best_estimator_xgb, 
        X_test, 
        y_test_enc, 
        random_state=42
    ).importances_mean  # get avg. importance score for each feature


df_feature_importances = pd.DataFrame(
    {"XGB":fi_mean.tolist()},
    index = X_test.columns.to_list(),
)

In [ ]:

## plot
plt.figure(figsize=(15, 25))  
sns.set_style("whitegrid", {"axes.grid": False})

df_feature_importances.plot.barh(width=0.6)
plt.tick_params(axis="x", which="major", labelsize=12)
plt.tick_params(axis="y", which="major", labelsize=12)
plt.xlabel("Importance")
plt.ylabel("")
plt.legend(fontsize=15, loc="upper right")
plt.title(f"Feature importances for type of water use", fontweight="bold", fontsize=16)

plt.tight_layout()

## SHAP - SHapley Additive exPlanations
In order to compute the shap values (i.e. feature importance scores) for XGB, we first have to initalize an `Explainer object`. The `explainer` calculates the shap values

**Note:** The shap values represent the impact of each feature value on the prediction, while the mean of the shap values represent the feature's importance (i.e. expressed as importance scores) 


<img src= https://shap.readthedocs.io/en/latest/_images/shap_header.png alt="Drawing" style="width: 800px;"/>


Uncomment the code below:


In [ ]:
import shap 

# fit the explainer
shap_explainer = shap.Explainer(best_estimator_xgb.predict, X_test)

### Tasks
- Calculate the importance of each feature in predicting builidng age, use for this the shap values returned from `shap_explainer` . Note: The returned shap values have following attributes: `.base_values, .values, .data` - the former are the observed target values ( in our case from the training set), the mid  are the shap values, and the latter is the data we passed to the `shap_explainer`
-  wait -  the calculation takes a while :)
- plot the feature importances (mean absolute importance score ) including the feature names, e.g. via `shap_importances = np.abs(shap_values.values).mean(axis=0)`. You can use either a built-in function from SHAP (`shap.plots.bar(shap_values)`) for the plot or calculate first the mean absolute importance scores and then plot them
- Write down the differences you observe between both feature importance methods




In [ ]:
shap_values = shap_explainer(X_test)



In [ ]:
## option 1
shap.plots.bar(shap_values)


In [ ]:
# print the JS visualization code to the notebook
shap.initjs()

In [ ]:
## option 2

## calculate the mean absolute shap values
shap_importances = np.abs(shap_values.values).mean(axis=0)

## add the importance scores along the respective feature names (pd.Series)
shap_fi = pd.Series(
        shap_importances, 
        index=X_test.columns
    ).sort_values()


## plot the averaged importance scores
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
shap_fi.plot.barh()
ax.set_xlabel("Importance scores")
ax.set_title("SHAP feature importance scores for predicting water usage types")


Answer: 
We see following things (compared to permutation-based feature importance)
* In SHAP the computation takes longer, even for medium sized datasets
* Permutation-based feature importance gives a quick overview about the global importance of each feature, while SHAP calculates first the local importance of each feature (e.g. the feature's impact for each single prediction ) and the averages the them to global importance scores.
* Partial dependence plots (PDPs) can be a good addition to permutation-based importance methods. PDPs can give some more insights similar as the local feature importances in SHAP

#### SHAP - more in-depth analysis

**Note:**: In case `shap.Explainer()` throws an ValueError (`ValueError: could not convert string to float:`) then use the latest release of SHAP from GitHub, which contains a fix of this issue. 

Run within notebook cell
!uv remove shap
!uv add "git+https://github.com/shap/shap.git@master"

-> then restart the kernel

In [ ]:
# Initialize JavaScript visualization - use Jupyter notebook to see the interactive features of the plots
shap.initjs()

In [ ]:
## get xgb object
xgb_trained_object = final_models_list[0].best_estimator_.named_steps["model"]#.get_booster()
xgb_trained_object



In [ ]:
masker = shap.maskers.Independent(data = X_train)
explainer = shap.TreeExplainer(xgb_trained_object, data=masker )
ev = explainer.expected_value
ev

In [ ]:
# This is what your XGB model would predict on average given background dataset (fed to explainer above):
xgb_trained_object.predict_proba(masker.data).mean(0)

In [ ]:
# shap force plot for the first prediction of the training set
# Here we want to interpret the output value for the 1st observation in our dataframe. 
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_train)
shap.force_plot(explainer.expected_value[0], shap_values[0], feature_names = explainer.data_feature_names)

# shap.plots.force(explainer.expected_value[1], shap_values.values[1,:], X_train.iloc[0, :])

In [ ]:
# ... and for the nth-observation in the dataframe
def shap_plot(clf, j):
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_train)
    # p = shap.force_plot(explainer.expected_value, shap_values[j], X_train.iloc[[j]], matplotlib = True, show = False)
    p = shap.force_plot(explainer.expected_value[j], shap_values[j], X_train.iloc[[j]], matplotlib = True, show = False)
    plt.savefig(f'tmp_record_j.svg')
    plt.close()
    return(p)

shap_plot(clf=xgb_trained_object, j=0)



In [ ]:
shap.summary_plot(shap_values, X_train)